# 6.6.– Métricas en NLP

Para **evaluar modelos de NLP**, no basta con mirar el *accuracy*.  
Las métricas deben **adaptarse a la tarea**: traducción, resumen, clasificación, generación, etc.

En este notebook exploramos las métricas más utilizadas en distintas tareas:

| Métrica | Uso | Descripción |
|----------|-----|--------------|
| **BLEU** | Traducción | Mide coincidencia entre la traducción generada y la referencia. |
| **ROUGE** | Resumen | Evalúa solapamiento de palabras o frases clave. |
| **Perplejidad** | Modelos de lenguaje | Cuantifica qué tan “sorprendido” está el modelo ante un texto. |
| **Cosine Similarity** | Embeddings | Evalúa similitud semántica entre vectores. |
| **Exact Match (EM)** | QA / Clasificación | Verifica si la respuesta coincide exactamente. |

In [22]:
# !pip install sentence_transformers

import re
import math
import torch
import numpy as np
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

## 1. BLEU – Traducción automática

El puntaje **BLEU (Bilingual Evaluation Understudy)** se utiliza para evaluar la calidad de una **traducción automática**.  
Compara cuántos fragmentos (*n-gramas*) de la traducción generada por el modelo coinciden con una traducción humana de referencia.

En este ejemplo usamos un modelo preentrenado de **Hugging Face** para traducir una frase del español al inglés, y luego medimos qué tan cercana es su traducción respecto a la versión humana usando el **BLEU score**.

In [23]:
# Modelo de traducción español → inglés
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")

texto_original = "El rápido avance de la inteligencia artificial está transformando la forma en que trabajamos y aprendemos."
traduccion_modelo = translator(texto_original)[0]['translation_text']

# Traducción humana de referencia
traduccion_humana = "The rapid progress of artificial intelligence is transforming the way we work and learn."

# Conversión a tokens
reference = [traduccion_humana.lower().split()]
candidate = traduccion_modelo.lower().split()

# Cálculo del BLEU
smooth = SmoothingFunction().method1
bleu = sentence_bleu(reference, candidate, smoothing_function=smooth)

print("\nTexto original (ES):", texto_original)
print("Traducción del modelo:", traduccion_modelo)
print("Traducción humana:", traduccion_humana)
print(f"\nPuntaje BLEU: {bleu:.2f}")

Device set to use cuda:0



Texto original (ES): El rápido avance de la inteligencia artificial está transformando la forma en que trabajamos y aprendemos.
Traducción del modelo: The rapid advance of artificial intelligence is transforming the way we work and learn.
Traducción humana: The rapid progress of artificial intelligence is transforming the way we work and learn.

Puntaje BLEU: 0.81


## 2. ROUGE – Resumen automático

Las métricas **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** se utilizan para evaluar la calidad de los **resúmenes automáticos**.  
Miden el solapamiento de palabras o frases clave entre el resumen generado por el modelo y un resumen de referencia escrito por humanos.

En este ejemplo usaremos un modelo preentrenado para generar un resumen de un texto, y luego compararemos su resultado con un resumen de referencia utilizando **ROUGE-1** y **ROUGE-L**.

In [24]:
# Modelo de resumen en inglés
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

texto = """
Artificial intelligence (AI) is rapidly changing the world. 
From healthcare to education, AI systems are being used to improve efficiency and decision-making. 
However, these advancements also raise ethical questions about data privacy, bias, and employment.
"""

# Resumen generado por el modelo (ajustado para evitar warnings)
resumen_modelo = summarizer(
    texto,
    max_new_tokens=28,
    min_length=15,
    do_sample=False
)[0]['summary_text']

# Resumen de referencia (humano)
resumen_referencia = "AI is transforming fields like healthcare and education but raises ethical concerns about privacy and bias."

# Calcular métricas ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score(resumen_referencia, resumen_modelo)

print("\nTexto original:", texto)
print("\nResumen generado:", resumen_modelo)
print("\nResumen humano:", resumen_referencia)
print(f"\nPuntaje ROUGE-1: {scores['rouge1'].fmeasure:.2f}")
print(f"Puntaje ROUGE-L: {scores['rougeL'].fmeasure:.2f}")

Device set to use cuda:0
Your max_length is set to 142, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)



Texto original: 
Artificial intelligence (AI) is rapidly changing the world. 
From healthcare to education, AI systems are being used to improve efficiency and decision-making. 
However, these advancements also raise ethical questions about data privacy, bias, and employment.


Resumen generado: Artificial intelligence (AI) is rapidly changing the world. From healthcare to education, AI systems are being used to improve efficiency and

Resumen humano: AI is transforming fields like healthcare and education but raises ethical concerns about privacy and bias.

Puntaje ROUGE-1: 0.27
Puntaje ROUGE-L: 0.27


## 3. Perplejidad – Modelos de lenguaje

La **perplejidad (perplexity)** mide qué tan “sorprendido” está un modelo de lenguaje ante una secuencia de texto.  
Cuanto menor es la perplejidad, mejor el modelo predice las palabras que aparecen en el texto.

En este ejemplo usaremos un modelo de lenguaje preentrenado para calcular la perplejidad de dos frases: una gramaticalmente correcta y otra con errores, para observar cómo cambia el valor.

In [25]:
# Modelo de lenguaje (GPT-2)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Frases de prueba
textos = [
    "The cat sat on the mat.",
    "Mat the on cat sat the."
]

for texto in textos:
    inputs = tokenizer(texto, return_tensors="pt")
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    perplexity = math.exp(loss)
    print(f"Frase: {texto}")
    print(f"Perplejidad: {perplexity:.2f}\n")

Frase: The cat sat on the mat.
Perplejidad: 90.24

Frase: Mat the on cat sat the.
Perplejidad: 2620.66



## 4. Similitud semántica – Distancia coseno

La **similitud del coseno** mide el grado de parecido entre dos textos en función de sus **representaciones vectoriales (embeddings)**.  
Su valor varía entre `-1` y `1`:

- `1` → Significa que los textos son semánticamente muy similares.  
- `0` → No hay relación semántica aparente.  
- `-1` → Son opuestos en significado.

En este ejemplo usamos un modelo de *Sentence Transformers* para generar embeddings y comparar pares de frases.

In [30]:
# Modelo de embeddings semánticos
model = SentenceTransformer("all-MiniLM-L6-v2")

# Frases a comparar
frases = [
    ("The cat is sleeping on the sofa.", "A cat is taking a nap on the couch."),
    ("The weather is sunny today.", "It will rain heavily tomorrow.")
]

for f1, f2 in frases:
    emb1 = model.encode(f1, convert_to_tensor=True)
    emb2 = model.encode(f2, convert_to_tensor=True)
    similitud = util.cos_sim(emb1, emb2).item()
    print(f"\nFrase 1: {f1}\nFrase 2: {f2}\nSimilitud coseno: {similitud:.3f}")


Frase 1: The cat is sleeping on the sofa.
Frase 2: A cat is taking a nap on the couch.
Similitud coseno: 0.784

Frase 1: The weather is sunny today.
Frase 2: It will rain heavily tomorrow.
Similitud coseno: 0.406


## 5. Exact Match (EM)

La métrica **Exact Match (EM)** se utiliza en tareas de **Question Answering (QA)** o **clasificación textual**.  
Evalúa si la respuesta generada por el modelo **coincide exactamente** con la respuesta correcta (después de normalizar texto, mayúsculas, tildes o puntuación).

Un valor de `EM = 1.0` indica coincidencia perfecta; `EM = 0.0` indica que la respuesta es diferente.

In [37]:
# Modelo de QA en inglés
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

contexto = """
The Eiffel Tower is one of the most famous landmarks in the world.
It is located in Paris, France, and was completed in 1889.
Every year, millions of tourists visit it to enjoy the view of the city.
"""

pregunta = "Where is the Eiffel Tower located?"
respuesta_real = "Paris, France"

# Respuesta del modelo
respuesta_modelo = qa_pipeline({"context": contexto, "question": pregunta})["answer"]

# Función de normalización para comparar textos
def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9 ]", "", texto)  # eliminar puntuación
    texto = texto.strip()
    return texto

# Calcular Exact Match (1 si coincide exactamente, 0 si no)
em = 1.0 if normalizar(respuesta_modelo) == normalizar(respuesta_real) else 0.0

print(f"\nPregunta: {pregunta}")
print(f"Respuesta real: {respuesta_real}")
print(f"Respuesta del modelo: {respuesta_modelo}")
print(f"\nExact Match (EM): {em:.1f}")

Device set to use cuda:0



Pregunta: Where is the Eiffel Tower located?
Respuesta real: Paris, France
Respuesta del modelo: Paris, France

Exact Match (EM): 1.0
